# Neo4j GraphRAG Python Package Feature Exploration

This notebook walks through the main features of the [`neo4j-graphrag`](https://pypi.org/project/neo4j-graphrag/) package using:
- **Neo4j Aura** as the graph database
- **Ollama** (local, free) for both LLM and embeddings
  - LLM: `granite4.2:3b` (IBM Granite 4.2, about 2.2 GB). Any Ollama chat model works: change `OLLAMA_LLM_MODEL` in section 2
  - Embeddings: `all-minilm` (384 dimensions)

**Prerequisites**
- Python 3.10 or later
- Ollama installed and running (`ollama serve`)
- `granite4.2:3b` and `all-minilm` pulled (`ollama pull granite4.2:3b && ollama pull all-minilm`)
- Neo4j Aura connection details (URI, username, password) in the environment variables `NEO4J_URI`, `NEO4J_USERNAME` and `NEO4J_PASSWORD`

> **Warning:** section 4 deletes everything in the target database, not just what this notebook creates. Use an empty or dedicated Aura instance.

**Sections**
1. Install dependencies
2. Configuration
3. Connect to Neo4j Aura
4. Load sample data
5. Create indexes (vector + full-text)
6. Embed sample data
7. Retriever: VectorRetriever
8. Retriever: VectorCypherRetriever
9. Retriever: HybridRetriever
10. Retriever: Text2CypherRetriever
11. GraphRAG pipeline
12. KG Builder: SimpleKGPipeline
13. Teardown

## 1. Install dependencies

In [1]:
# Core package + Ollama extras + experimental KG builder

%pip install "neo4j-graphrag[ollama,experimental]==1.21.0" \
             neo4j==6.1.0 --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import httpx
import os

from neo4j import GraphDatabase, NotificationDisabledClassification
from neo4j_graphrag.embeddings import OllamaEmbeddings
from neo4j_graphrag.exceptions import LLMGenerationError
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline
from neo4j_graphrag.generation import GraphRAG, RagTemplate
from neo4j_graphrag.indexes import create_vector_index, create_fulltext_index
from neo4j_graphrag.llm import OllamaLLM
from neo4j_graphrag.llm.types import LLMResponse
from neo4j_graphrag.retrievers import HybridRetriever, Text2CypherRetriever, VectorCypherRetriever, VectorRetriever
from neo4j_graphrag.types import RetrieverResultItem

## 2. Configuration

In [3]:
# Neo4j Aura connection
NEO4J_URI      = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]

# Ollama models
OLLAMA_LLM_MODEL     = "granite4.2:3b"  # chat / generation model (IBM Granite 4.2, 3B)
OLLAMA_EMBED_MODEL   = "all-minilm"     # embedding model
EMBEDDING_DIMENSIONS = 384              # all-minilm produces 384-dim vectors

# Model used only for knowledge graph extraction (section 12). Defaults to the same model;
# switch to "granite4.2:8b" (ollama pull granite4.2:8b) if the 3B model struggles with extraction.
OLLAMA_KG_MODEL = OLLAMA_LLM_MODEL

# Index names (used throughout the notebook)
VECTOR_INDEX_NAME   = "graphrag_vector_idx"
FULLTEXT_INDEX_NAME = "graphrag_fulltext_idx"

# Node label used for sample data
NODE_LABEL = "Article"

print("Credentials set.")

Credentials set.


**A note on `OllamaLLMNoThink`.** Granite 4.2 reasons before it answers by default and in testing a one-line extraction took about 18 seconds and 774 output tokens with thinking on, against 1.5 seconds and 63 tokens with it off. `neo4j-graphrag` 1.21.0 pins an `ollama` client that has no `think` option and its async methods (the ones the KG builder uses) pass the whole `model_params` dict as Ollama's `options`, so `temperature` and `format` are silently ignored there. The small subclass below calls Ollama's HTTP API directly to fix both. If you use a model without a thinking mode, plain `OllamaLLM` works.

In [4]:
# Ollama LLM wrapper that can switch off "thinking" mode (see the note above).

class OllamaLLMNoThink(OllamaLLM):
    """OllamaLLM that calls Ollama's HTTP API directly so that `think` can be switched off,
    and so that `options` and `format` in model_params are honored on the async path too."""

    def __init__(self, *args, think=False, host=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.think = think
        host = host or os.environ.get("OLLAMA_HOST") or "http://localhost:11434"
        self.host = host if "://" in host else f"http://{host}"

    def _payload(self, input, message_history=None, system_instruction=None):
        if isinstance(input, str):
            messages = self.get_messages(input, message_history, system_instruction)
        else:
            messages = self.get_messages_v2(input)
        messages = [m.model_dump(exclude_none=True) if hasattr(m, "model_dump") else m for m in messages]
        payload = {
            "model": self.model_name,
            "messages": messages,
            "stream": False,
            "think": self.think,
            "options": self.model_params.get("options", {}),
        }
        if "format" in self.model_params:
            payload["format"] = self.model_params["format"]
        return payload

    @staticmethod
    def _parse(response):
        try:
            response.raise_for_status()
            return LLMResponse(content=response.json()["message"]["content"] or "")
        except Exception as exc:
            raise LLMGenerationError(exc)

    def invoke(self, input, message_history=None, system_instruction=None, response_format=None, **kwargs):
        if response_format is not None:
            return super().invoke(input, message_history, system_instruction, response_format, **kwargs)
        payload = self._payload(input, message_history, system_instruction)
        return self._parse(httpx.post(f"{self.host}/api/chat", json=payload, timeout=300))

    async def ainvoke(self, input, message_history=None, system_instruction=None, response_format=None, **kwargs):
        if response_format is not None:
            return await super().ainvoke(input, message_history, system_instruction, response_format, **kwargs)
        payload = self._payload(input, message_history, system_instruction)
        async with httpx.AsyncClient(timeout=300) as client:
            return self._parse(await client.post(f"{self.host}/api/chat", json=payload))


## 3. Connect to Neo4j Aura

In [5]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth = (NEO4J_USERNAME, NEO4J_PASSWORD),
    notifications_disabled_classifications=[NotificationDisabledClassification.DEPRECATION]
)

print(driver.verify_connectivity())  # None is expected
print("Connection created.")

None
Connection created.


## 4. Load sample data

We start with a full database reset — dropping all indexes, constraints and data — so the notebook always runs against a clean slate. This affects the whole database, so don't run it against an instance that holds data you want to keep.

We then load a small set of short articles about graph databases and related topics. Each node gets an `id`, a `title`, a `text` and a `category` property. The `text` property is what we'll embed and search over.

In [6]:
# Full database reset
with driver.session() as session:
    # Drop all constraints first — each constraint also drops its backing index
    constraints = session.run("SHOW CONSTRAINTS YIELD name").data()
    for con in constraints:
        session.run(f"DROP CONSTRAINT `{con['name']}` IF EXISTS")
    print(f"Dropped {len(constraints)} constraint(s).")

    # Drop any remaining standalone indexes (not backed by a constraint)
    indexes = session.run(
        "SHOW INDEXES YIELD name, owningConstraint "
        "WHERE owningConstraint IS NULL"
    ).data()
    for idx in indexes:
        session.run(f"DROP INDEX `{idx['name']}` IF EXISTS")
    print(f"Dropped {len(indexes)} standalone index(es).")

    # Delete all nodes and relationships
    session.run("MATCH (n) DETACH DELETE n")
    print("All nodes and relationships deleted.")

print("Database fully reset.")

Dropped 0 constraint(s).
Dropped 0 standalone index(es).
All nodes and relationships deleted.
Database fully reset.


In [7]:
SAMPLE_ARTICLES = [
    {
        "id": "art-001",
        "title": "Introduction to Graph Databases",
        "text": (
            "Graph databases store data as nodes and relationships rather than tables and rows. "
            "Neo4j is the most popular graph database and uses the Cypher query language. "
            "They excel at traversing highly connected data such as social networks, "
            "recommendation engines and fraud detection systems."
        ),
        "category": "databases",
    },
    {
        "id": "art-002",
        "title": "What is RAG?",
        "text": (
            "Retrieval Augmented Generation (RAG) combines a retrieval step with an LLM generation step. "
            "A query is first used to retrieve relevant documents from a knowledge base, "
            "and those documents are then passed as context to a large language model "
            "so it can produce a grounded, accurate answer."
        ),
        "category": "AI",
    },
    {
        "id": "art-003",
        "title": "GraphRAG: Combining Graphs and LLMs",
        "text": (
            "GraphRAG extends standard RAG by using a knowledge graph as the retrieval source. "
            "Instead of flat document chunks, the retriever can traverse graph relationships "
            "to gather richer, more connected context before passing it to the LLM. "
            "Neo4j's GraphRAG Python package makes this straightforward to implement."
        ),
        "category": "AI",
    },
    {
        "id": "art-004",
        "title": "Vector Indexes in Neo4j",
        "text": (
            "Neo4j supports native vector indexes for approximate nearest neighbor search. "
            "Embeddings are stored as properties on nodes or relationships. "
            "The index supports cosine and Euclidean similarity functions, "
            "and can be queried using the db.index.vector.queryNodes procedure or via the GraphRAG package."
        ),
        "category": "databases",
    },
    {
        "id": "art-005",
        "title": "Cypher Query Language Basics",
        "text": (
            "Cypher is Neo4j's declarative graph query language. "
            "Nodes are written in parentheses, e.g. (n:Person), "
            "and relationships in square brackets, e.g. -[r:KNOWS]->. "
            "The MATCH clause finds patterns, RETURN specifies what to output, "
            "and WHERE filters results. Cypher makes graph traversals intuitive to read and write."
        ),
        "category": "databases",
    },
    {
        "id": "art-006",
        "title": "Large Language Models Explained",
        "text": (
            "Large language models (LLMs) are neural networks trained on vast amounts of text "
            "to predict and generate human-like language. "
            "They power applications such as chatbots, code generation, summarization and question answering. "
            "Popular LLMs include GPT-4, Claude, Llama and Mistral."
        ),
        "category": "AI",
    },
    {
        "id": "art-007",
        "title": "Knowledge Graph Construction",
        "text": (
            "A knowledge graph organizes information as entities (nodes) and their relationships (edges). "
            "LLMs can extract entities and relationships from unstructured text automatically. "
            "The resulting graph can then be queried, visualized and used as a retrieval source for GraphRAG pipelines."
        ),
        "category": "AI",
    },
    {
        "id": "art-008",
        "title": "Neo4j Aura: Managed Cloud Database",
        "text": (
            "Neo4j Aura is a fully managed cloud graph database service. "
            "It removes the need to install and maintain Neo4j on your own infrastructure. "
            "Aura supports automatic backups, scaling and security out of the box, "
            "and connects via the standard Bolt protocol using the Neo4j Python driver."
        ),
        "category": "databases",
    },
]

# Write nodes to Neo4j
with driver.session() as session:
    for article in SAMPLE_ARTICLES:
        session.run(
            f"""
            CREATE (a:{NODE_LABEL} {{
                id: $id,
                title: $title,
                text: $text,
                category: $category
            }})
            """,
            **article,
        )

print(f"Loaded {len(SAMPLE_ARTICLES)} {NODE_LABEL} nodes into Neo4j.")

Loaded 8 Article nodes into Neo4j.


## 5. Create indexes

We need two indexes:
- **Vector index** — for semantic similarity search (VectorRetriever, VectorCypherRetriever and the vector half of HybridRetriever)
- **Full-text index** — for keyword search (the other half of HybridRetriever)

In [8]:
# Vector index on Article.embedding
create_vector_index(
    driver,
    VECTOR_INDEX_NAME,
    label=NODE_LABEL,
    embedding_property="embedding",
    dimensions=EMBEDDING_DIMENSIONS,
    similarity_fn="cosine",
)
print(f"Vector index '{VECTOR_INDEX_NAME}' created.")

# Full-text index on Article.title and Article.text
create_fulltext_index(
    driver,
    FULLTEXT_INDEX_NAME,
    label=NODE_LABEL,
    node_properties=["title", "text"],
)
print(f"Full-text index '{FULLTEXT_INDEX_NAME}' created.")

Vector index 'graphrag_vector_idx' created.
Full-text index 'graphrag_fulltext_idx' created.


## 6. Embed sample data

We use `OllamaEmbeddings` to generate a 384-dim vector for each article's `text` property and write it back to the node as the `embedding` property. Finally, we wait for the indexes to finish populating before we query them.

In [9]:
embedder = OllamaEmbeddings(model=OLLAMA_EMBED_MODEL)

with driver.session() as session:
    for article in SAMPLE_ARTICLES:
        vector = embedder.embed_query(article["text"])
        session.run(
            f"MATCH (a:{NODE_LABEL} {{id: $id}}) SET a.embedding = $embedding",
            id=article["id"],
            embedding=vector,
        )
        print(f"  Embedded: {article['title']}")

    # Make sure the indexes are online and populated before we query them
    session.run("CALL db.awaitIndexes(300)").consume()

print(f"\nAll {len(SAMPLE_ARTICLES)} articles embedded.")

  Embedded: Introduction to Graph Databases
  Embedded: What is RAG?
  Embedded: GraphRAG: Combining Graphs and LLMs
  Embedded: Vector Indexes in Neo4j
  Embedded: Cypher Query Language Basics
  Embedded: Large Language Models Explained
  Embedded: Knowledge Graph Construction
  Embedded: Neo4j Aura: Managed Cloud Database

All 8 articles embedded.


## 7. Retriever: VectorRetriever

The simplest retriever. It embeds the query, searches the vector index and returns the top-k most semantically similar nodes.

By default, each result's content is the node's properties rendered as a string. We pass a small `result_formatter` so the content reads as `title: text` and the title, category and score are kept as metadata. We reuse it for the hybrid retriever later.

In [10]:
def format_article(record):
    """Turn a search record into readable context, keeping the title and score as metadata."""
    node = record["node"]
    return RetrieverResultItem(
        content=f"{node['title']}: {node['text']}",
        metadata={"title": node["title"], "category": node["category"], "score": record["score"]},
    )

vector_retriever = VectorRetriever(
    driver=driver,
    index_name=VECTOR_INDEX_NAME,
    embedder=embedder,
    return_properties=["title", "text", "category"],
    result_formatter=format_article,
)

query = "How does semantic search work in a graph database?"
results = vector_retriever.search(query_text=query, top_k=3)

print(f"Query: {query}\n")
for i, item in enumerate(results.items, start=1):
    print(f"Result {i}: {item.metadata['title']} (score {item.metadata['score']:.3f})")

Query: How does semantic search work in a graph database?

Result 1: Knowledge Graph Construction (score 0.770)
Result 2: Introduction to Graph Databases (score 0.770)
Result 3: Cypher Query Language Basics (score 0.745)


## 8. Retriever: VectorCypherRetriever

Adds a custom Cypher traversal on top of the vector search. After the similarity search finds matching nodes, the `retrieval_query` is run to follow relationships and gather richer context.

In the Cypher below, `node` is the matched `Article` node and `score` is its similarity score — both are injected automatically.

To give the traversal something real to follow, we add `BUILDS_ON` relationships that reflect what each article actually says: `(a)-[:BUILDS_ON]->(b)` means article `a` depends on concepts introduced in article `b`. For example, GraphRAG extends standard RAG and RAG passes retrieved documents to an LLM. Following these links surfaces prerequisites that a similarity search alone won't rank highly. For each query, we compare what vector search returns on its own with what the graph adds.

In [11]:
# Relationships that reflect what the articles actually say.
# (a)-[:BUILDS_ON]->(b) means article a depends on concepts introduced in article b.
BUILDS_ON = [
    ("art-002", "art-006"),  # RAG passes retrieved documents to an LLM
    ("art-003", "art-002"),  # GraphRAG extends standard RAG
    ("art-003", "art-007"),  # GraphRAG uses a knowledge graph as its retrieval source
    ("art-003", "art-004"),  # the GraphRAG package queries Neo4j vector indexes
    ("art-007", "art-006"),  # LLMs extract the entities and relationships
    ("art-007", "art-001"),  # a knowledge graph is nodes and relationships
    ("art-004", "art-001"),  # vector indexes live inside a graph database
    ("art-005", "art-001"),  # Cypher is the graph database's query language
    ("art-008", "art-001"),  # Aura is a managed graph database
]

with driver.session() as session:
    session.run(
        f"""
        UNWIND $pairs AS pair
        MATCH (a:{NODE_LABEL} {{id: pair[0]}}), (b:{NODE_LABEL} {{id: pair[1]}})
        MERGE (a)-[:BUILDS_ON]->(b)
        """,
        pairs=[list(pair) for pair in BUILDS_ON],
    )
print(f"Created {len(BUILDS_ON)} BUILDS_ON relationships.\n")

# Retrieval query: the matched article, what it builds on, what builds on it,
# and everything it depends on up to three hops back (distinct titles)
retrieval_cypher = """
RETURN node.title AS title,
       node.text  AS text,
       score,
       [ (node)-[:BUILDS_ON]->(p) | p.title ] AS builds_on,
       [ (node)<-[:BUILDS_ON]-(d) | d.title ] AS built_on_by,
       COLLECT { MATCH (node)-[:BUILDS_ON*1..3]->(p) RETURN DISTINCT p.title } AS all_prerequisites
"""

def format_with_context(record):
    """Readable context for the LLM, with the graph relationships spelled out."""
    lines = [f"{record['title']}: {record['text']}"]
    if record["builds_on"]:
        lines.append("Prerequisites (this article builds on): " + ", ".join(record["builds_on"]))
    if record["built_on_by"]:
        lines.append("Later articles that depend on this one (not prerequisites): " + ", ".join(record["built_on_by"]))
    further = [t for t in record["all_prerequisites"] if t not in record["builds_on"]]
    if further:
        lines.append("Deeper prerequisites: " + ", ".join(further))
    return RetrieverResultItem(
        content="\n".join(lines),
        metadata={
            "title": record["title"],
            "score": record["score"],
            "builds_on": record["builds_on"],
            "built_on_by": record["built_on_by"],
            "all_prerequisites": record["all_prerequisites"],
        },
    )

vector_cypher_retriever = VectorCypherRetriever(
    driver=driver,
    index_name=VECTOR_INDEX_NAME,
    embedder=embedder,
    retrieval_query=retrieval_cypher,
    result_formatter=format_with_context,
)

for query in [
    "What do I need to understand before using GraphRAG?",
    "What is a knowledge graph and how is it built?",
]:
    vector_only = vector_retriever.search(query_text=query, top_k=3)
    with_graph = vector_cypher_retriever.search(query_text=query, top_k=3)

    returned = [item.metadata["title"] for item in vector_only.items]
    print(f"Query: {query}\n")
    print("Vector search alone returned:")
    for title in returned:
        print(f"  - {title}")

    print("\nWith graph context:")
    surfaced = set()
    for item in with_graph.items:
        m = item.metadata
        print(f"  - {m['title']} (score {m['score']:.3f})")
        print(f"      builds on   : {m['builds_on'] or '-'}")
        print(f"      built on by : {m['built_on_by'] or '-'}")
        print(f"      all prereqs : {m['all_prerequisites'] or '-'}")
        # Count only prerequisites (what this article depends on), not articles that depend on it
        surfaced |= set(m["builds_on"]) | set(m["all_prerequisites"])

    extra = sorted(surfaced - set(returned))
    print(f"\nPrerequisites the graph surfaced that vector search alone did not return: {extra}\n")

Created 9 BUILDS_ON relationships.

Query: What do I need to understand before using GraphRAG?

Vector search alone returned:
  - Knowledge Graph Construction
  - GraphRAG: Combining Graphs and LLMs
  - Introduction to Graph Databases

With graph context:
  - Knowledge Graph Construction (score 0.781)
      builds on   : ['Introduction to Graph Databases', 'Large Language Models Explained']
      built on by : ['GraphRAG: Combining Graphs and LLMs']
      all prereqs : ['Introduction to Graph Databases', 'Large Language Models Explained']
  - GraphRAG: Combining Graphs and LLMs (score 0.778)
      builds on   : ['What is RAG?', 'Vector Indexes in Neo4j', 'Knowledge Graph Construction']
      built on by : -
      all prereqs : ['What is RAG?', 'Vector Indexes in Neo4j', 'Knowledge Graph Construction', 'Large Language Models Explained', 'Introduction to Graph Databases']
  - Introduction to Graph Databases (score 0.751)
      builds on   : -
      built on by : ['Vector Indexes in Neo4j

## 9. Retriever: HybridRetriever

Combines vector (semantic) search with full-text (keyword) search. The package normalizes each list's scores by dividing by that list's top score, then keeps the higher score for each node (the default `naive` ranker; a `linear` ranker that weights the two lists is also available). A score of 1.0 therefore means the node topped at least one of the two lists. This is useful when exact keyword matches matter as much as semantic similarity. We run three queries against both the hybrid retriever and the vector index alone, to see what the full-text half changes. The last two queries use exact terms that may or may not appear in the sample articles; if the two rankings come out the same, that is a result worth reporting.

In [12]:
hybrid_retriever = HybridRetriever(
    driver=driver,
    vector_index_name=VECTOR_INDEX_NAME,
    fulltext_index_name=FULLTEXT_INDEX_NAME,
    embedder=embedder,
    return_properties=["title", "text", "category"],
    result_formatter=format_article,
)

# One natural-language query, then two keyword-heavy queries where exact terms should matter
for query in [
    "Cypher query language Neo4j",
    "Euclidean similarity",
    "Bolt protocol",
]:
    hybrid_results = hybrid_retriever.search(query_text=query, top_k=3)
    vector_results = vector_retriever.search(query_text=query, top_k=3)

    print(f"Query: {query}\n")
    print("Hybrid search:")
    for i, item in enumerate(hybrid_results.items, start=1):
        print(f"  {i}. {item.metadata['title']} (score {item.metadata['score']:.3f})")

    print("\nVector search only:")
    for i, item in enumerate(vector_results.items, start=1):
        print(f"  {i}. {item.metadata['title']} (score {item.metadata['score']:.3f})")
    print()


Query: Cypher query language Neo4j

Hybrid search:
  1. Cypher Query Language Basics (score 1.000)
  2. Introduction to Graph Databases (score 0.916)
  3. Vector Indexes in Neo4j (score 0.895)

Vector search only:
  1. Cypher Query Language Basics (score 0.885)
  2. Introduction to Graph Databases (score 0.811)
  3. Vector Indexes in Neo4j (score 0.792)

Query: Euclidean similarity

Hybrid search:
  1. Vector Indexes in Neo4j (score 1.000)
  2. Knowledge Graph Construction (score 0.852)
  3. Introduction to Graph Databases (score 0.803)

Vector search only:
  1. Vector Indexes in Neo4j (score 0.680)
  2. Knowledge Graph Construction (score 0.579)
  3. Introduction to Graph Databases (score 0.546)

Query: Bolt protocol

Hybrid search:
  1. Neo4j Aura: Managed Cloud Database (score 1.000)
  2. Knowledge Graph Construction (score 0.933)
  3. What is RAG? (score 0.901)

Vector search only:
  1. Neo4j Aura: Managed Cloud Database (score 0.640)
  2. Knowledge Graph Construction (score 0.597)

## 10. Retriever: Text2CypherRetriever

Translates a natural language question into a Cypher query using an LLM, then runs that query against Neo4j, after checking that it is read-only. We provide a schema description so the LLM knows what nodes and properties are available.

We try three questions: one close to our examples, one that needs an aggregation, and one that needs the `BUILDS_ON` relationship. The retriever returns the generated Cypher in `results.metadata`, so we can see what the LLM wrote. Small models don't always get this right, so failures are caught and printed.

We run the questions twice, once with Granite's thinking mode on (its default) and once with it off. In our testing, this made a clear difference to the generated Cypher.

In [13]:
# Same model, same settings, except for thinking mode. Temperature 0 makes repeated runs more consistent.
llm_thinking = OllamaLLMNoThink(model_name=OLLAMA_LLM_MODEL, model_params={"options": {"temperature": 0}}, think=True)
llm = OllamaLLMNoThink(model_name=OLLAMA_LLM_MODEL, model_params={"options": {"temperature": 0}}, think=False)  # used from here on

# Describe the schema to help the LLM generate correct Cypher
neo4j_schema = """
Node labels:
  - Article { id: STRING, title: STRING, text: STRING, category: STRING }

Relationship types:
  - (Article)-[:BUILDS_ON]->(Article)   # the first article depends on concepts introduced in the second
"""

# A few examples help the LLM produce better Cypher
examples = [
    "USER: How many articles are there?\nCYPHER: MATCH (a:Article) RETURN count(a) AS total",
    "USER: List all article titles\nCYPHER: MATCH (a:Article) RETURN a.title AS title",
    "USER: Find AI articles\nCYPHER: MATCH (a:Article {category: 'AI'}) RETURN a.title AS title, a.text AS text",
]

questions = [
    "Which articles are about AI?",                                         # close to an example
    "How many articles are in each category?",                              # needs an aggregation
    "Which articles does 'GraphRAG: Combining Graphs and LLMs' build on?",  # needs the relationship
]

for label, model in [("thinking on", llm_thinking), ("thinking off", llm)]:
    t2c_retriever = Text2CypherRetriever(
        driver=driver,
        llm=model,
        neo4j_schema=neo4j_schema,
        examples=examples,
    )
    print(f"===== {label} =====\n")
    for query in questions:
        print(f"Query: {query}")
        try:
            results = t2c_retriever.search(query_text=query)
            print(f"  Generated Cypher: {results.metadata['cypher']}")
            for item in results.items:
                print(f"  {item.content}")
        except Exception as exc:
            print(f"  Failed: {type(exc).__name__}: {str(exc)[:200]}")
        print()


===== thinking on =====

Query: Which articles are about AI?
  Generated Cypher: MATCH (a:Article {category: 'AI'}) RETURN a.title AS title, a.text AS text
  <Record title='What is RAG?' text='Retrieval Augmented Generation (RAG) combines a retrieval step with an LLM generation step. A query is first used to retrieve relevant documents from a knowledge base, and those documents are then passed as context to a large language model so it can produce a grounded, accurate answer.'>
  <Record title='GraphRAG: Combining Graphs and LLMs' text="GraphRAG extends standard RAG by using a knowledge graph as the retrieval source. Instead of flat document chunks, the retriever can traverse graph relationships to gather richer, more connected context before passing it to the LLM. Neo4j's GraphRAG Python package makes this straightforward to implement.">
  <Record title='Large Language Models Explained' text='Large language models (LLMs) are neural networks trained on vast amounts of text to predict a

## 11. GraphRAG pipeline

The `GraphRAG` class wires a retriever and an LLM together into a complete question-answering pipeline. The retriever gathers context from the graph; the LLM uses that context to generate a grounded answer.

We first try it with the `VectorRetriever`, then ask one question through all three similarity-based retrievers to see how the retrieved context changes the answer and finally override the prompt.

In [14]:
# 11a. GraphRAG with VectorRetriever
rag_vector = GraphRAG(retriever=vector_retriever, llm=llm)

question = "What is GraphRAG and why is it useful?"
response = rag_vector.search(query_text=question, retriever_config={"top_k": 3})

print("=== GraphRAG with VectorRetriever ===")
print(f"Question: {question}")
print(f"Answer  : {response.answer}")
print()

=== GraphRAG with VectorRetriever ===
Question: What is GraphRAG and why is it useful?
Answer  : GraphRAG is a method that combines graph databases (like Neo4j) with large language models (LLMs) to improve information retrieval and understanding. It uses a knowledge graph—where data is stored as nodes (entities) and edges (relationships)—as the source for retrieval, rather than flat document chunks. This allows the retriever to traverse graph relationships, gathering richer, more connected context before passing it to the LLM.

It is useful because:
- It enables richer, more contextually relevant answers by leveraging the structure of relationships between entities (e.g., in social networks, recommendation systems, or fraud detection).
- It improves accuracy and relevance compared to traditional Retrieval-Augmented Generation (RAG) that only uses isolated document snippets.
- It supports complex reasoning and deeper understanding through graph traversal, which is especially valuable in

In [15]:
# 11b. The same question through three retrievers
question = "What do I need to understand before using GraphRAG?"

for name, retriever in [
    ("VectorRetriever", vector_retriever),
    ("HybridRetriever", hybrid_retriever),
    ("VectorCypherRetriever", vector_cypher_retriever),
]:
    rag = GraphRAG(retriever=retriever, llm=llm)
    response = rag.search(query_text=question, retriever_config={"top_k": 3})
    print(f"=== GraphRAG with {name} ===")
    print(f"Question: {question}")
    print(f"Answer  : {response.answer}")
    print()

=== GraphRAG with VectorRetriever ===
Question: What do I need to understand before using GraphRAG?
Answer  : To use GraphRAG effectively, you should understand the following key concepts:

1. **Knowledge Graph Construction** – Know how entities (nodes) and their relationships (edges) are organized in a knowledge graph, and how LLMs can automatically extract these from unstructured text.

2. **Graph Databases and Neo4j** – Understand that GraphRAG relies on a graph database (e.g., Neo4j) that stores data using nodes and relationships via Cypher queries. Familiarity with basic graph concepts (nodes, edges, traversal) is helpful.

3. **GraphRAG Pipeline** – Grasp that GraphRAG extends standard Retrieval-Augmented Generation (RAG) by using the knowledge graph as the retrieval source. Instead of retrieving flat document chunks, the retriever traverses graph relationships to gather richer, more connected context before feeding it to the LLM.

With these foundations, you’ll be able to use Ne

In [16]:
# 11c. Custom prompt template
# You can override the default prompt to change how the LLM uses the retrieved context.

custom_template = RagTemplate(
    template="""
You are a helpful assistant that answers questions about graph databases and AI.
Use only the context below to answer. If the context doesn't contain enough information,
say so honestly. Keep your answer concise — two or three sentences at most.

Context:
{context}

Question: {query_text}

Answer:""",
    expected_inputs=["context", "query_text"],
)

rag_custom = GraphRAG(retriever=vector_retriever, llm=llm, prompt_template=custom_template)

question = "What query language does Neo4j use and what makes it readable?"
response = rag_custom.search(query_text=question, retriever_config={"top_k": 2})

print("=== GraphRAG with custom prompt ===")
print(f"Question: {question}")
print(f"Answer  : {response.answer}")

=== GraphRAG with custom prompt ===
Question: What query language does Neo4j use and what makes it readable?
Answer  : Neo4j uses the Cypher query language. Cypher makes graph traversals intuitive to read and write.


## 12. KG Builder: SimpleKGPipeline

`SimpleKGPipeline` uses an LLM to extract entities and relationships from unstructured text and writes them as nodes and relationships into Neo4j.

We define the node types, relationship types and allowed patterns up front so the LLM knows what schema to extract into.

The extraction prompt asks the LLM for JSON. We use a separate `OllamaLLMNoThink` (section 2) with thinking off and the temperature set to 0. Thinking mode made extraction very slow and our first run, before the fix, logged "LLM response is not valid JSON" and extracted nothing. We haven't confirmed that thinking mode caused the invalid JSON. If extraction returns nothing, set `OLLAMA_KG_MODEL` in section 2 to a larger model such as `granite4.2:8b`.

> **Note:** This uses the `experimental` namespace and runs asynchronously. It reuses the `embedder` from section 6, so run that cell first.

In [17]:
# A separate LLM for extraction: thinking off, temperature 0 (the package repairs slightly malformed JSON)
kg_llm = OllamaLLMNoThink(
    model_name=OLLAMA_KG_MODEL,
    model_params={"options": {"temperature": 0}},
)

# Schema: what the LLM should look for in the text
node_types = ["Technology", "Company", "Concept"]
relationship_types = ["USES", "DEVELOPED_BY", "RELATED_TO", "PART_OF", "SERVES"]
patterns = [
    ("Technology", "DEVELOPED_BY", "Company"),
    ("Technology", "USES", "Technology"),
    ("Technology", "SERVES", "Technology"),
    ("Technology", "RELATED_TO", "Concept"),
    ("Concept", "PART_OF", "Concept"),
]

kg_builder = SimpleKGPipeline(
    llm=kg_llm,
    driver=driver,
    embedder=embedder,
    schema={
        "node_types": node_types,
        "relationship_types": relationship_types,
        "patterns": patterns,
    },
    on_error="IGNORE",
    from_file=False,
)

# Sample text to extract a knowledge graph from
source_text = """
Neo4j is a graph database developed by Neo4j, Inc.
It uses the Cypher query language to store and retrieve data as nodes and relationships.
Neo4j supports vector indexes, making it suitable for GraphRAG applications.
GraphRAG is a concept that combines knowledge graphs with retrieval augmented generation.
Ollama is a tool that lets developers run large language models locally on their own hardware.
LLMs such as Llama and Mistral can be served by Ollama and used for text generation and entity extraction.
"""

print("Running SimpleKGPipeline — this may take a moment...")
await kg_builder.run_async(text=source_text)
print("Knowledge graph extraction complete.")

Running SimpleKGPipeline — this may take a moment...
Knowledge graph extraction complete.


In [18]:
def domain_labels(labels):
    """Hide the labels the KG builder adds internally."""
    return [l for l in labels if not l.startswith("__")] or labels

# Inspect what was extracted
with driver.session() as session:
    # Nodes created by the KG builder
    result = session.run(
        "MATCH (n) WHERE n:Technology OR n:Company OR n:Concept "
        "RETURN labels(n) AS labels, n.name AS name LIMIT 20"
    )
    nodes = result.data()

    # Relationships created by the KG builder
    result = session.run(
        "MATCH (a)-[r]->(b) "
        "WHERE a:Technology OR a:Company OR a:Concept "
        "RETURN labels(a) AS from_labels, a.name AS from_name, "
        "       type(r) AS rel, labels(b) AS to_labels, b.name AS to_name "
        "LIMIT 20"
    )
    rels = result.data()

if not nodes:
    print("No entities were extracted. With on_error='IGNORE' the pipeline swallows failures, so check the cell above for\n"
          "'LLM response is not valid JSON' messages. Try a larger OLLAMA_KG_MODEL (section 2) and re-run section 12.\n")

print(f"Extracted {len(nodes)} nodes:")
for n in nodes:
    print(f"  {domain_labels(n['labels'])} — {n['name']}")

print(f"\nExtracted {len(rels)} relationships:")
for r in rels:
    to_name = r["to_name"] or "(source chunk)"
    print(f"  ({domain_labels(r['from_labels'])}: {r['from_name']}) -[{r['rel']}]-> ({domain_labels(r['to_labels'])}: {to_name})")

Extracted 8 nodes:
  ['Company'] — Neo4j, Inc.
  ['Technology'] — graph database
  ['Technology'] — Cypher query language
  ['Concept'] — GraphRAG
  ['Technology'] — vector indexes
  ['Company'] — Ollama
  ['Technology'] — large language model
  ['Concept'] — LLMs

Extracted 12 relationships:
  (['Company']: Neo4j, Inc.) -[FROM_CHUNK]-> (['Chunk']: (source chunk))
  (['Technology']: graph database) -[DEVELOPED_BY]-> (['Company']: Neo4j, Inc.)
  (['Technology']: graph database) -[USES]-> (['Technology']: Cypher query language)
  (['Technology']: graph database) -[FROM_CHUNK]-> (['Chunk']: (source chunk))
  (['Technology']: Cypher query language) -[FROM_CHUNK]-> (['Chunk']: (source chunk))
  (['Concept']: GraphRAG) -[FROM_CHUNK]-> (['Chunk']: (source chunk))
  (['Concept']: GraphRAG) -[PART_OF]-> (['Concept']: GraphRAG)
  (['Technology']: vector indexes) -[FROM_CHUNK]-> (['Chunk']: (source chunk))
  (['Company']: Ollama) -[FROM_CHUNK]-> (['Chunk']: (source chunk))
  (['Technology']: larg

## 13. Teardown

Run this cell to remove all indexes, constraints and data in the database, including everything this notebook created. Skip it if you want to keep the data in Aura for further exploration.

In [19]:
with driver.session() as session:
    # Drop constraints first (each also drops its backing index)
    constraints = session.run("SHOW CONSTRAINTS YIELD name").data()
    for con in constraints:
        session.run(f"DROP CONSTRAINT `{con['name']}` IF EXISTS")
    print(f"Dropped {len(constraints)} constraint(s).")

    # Drop remaining standalone indexes
    indexes = session.run(
        "SHOW INDEXES YIELD name, owningConstraint "
        "WHERE owningConstraint IS NULL"
    ).data()
    for idx in indexes:
        session.run(f"DROP INDEX `{idx['name']}` IF EXISTS")
    print(f"Dropped {len(indexes)} standalone index(es).")

    # Delete all nodes and relationships
    session.run("MATCH (n) DETACH DELETE n")
    print("All nodes and relationships deleted.")

Dropped 0 constraint(s).
Dropped 3 standalone index(es).
All nodes and relationships deleted.


In [20]:
driver.close()
print("Connections closed.")

Connections closed.
